# P5b — Fine-tune PanDerm Base phân loại (ISIC 2017 Task 3) trên Colab

Chọn **Select Kernel → Colab → Auto Connect** với runtime GPU (T4). Chạy tuần tự các cell.

- Train bằng `isic2017_cls_trainphase.csv`: split `test` của file này là bản sao `val`, nên test thật không bao giờ bị mở.
- `PILOT = True` chạy 1 epoch để đo VRAM, thời gian mỗi epoch và kiểm tra checkpoint/resume. Kết quả pilot không dùng để báo cáo.
- Checkpoint theo epoch và `checkpoint-best.pth` ghi thẳng vào Drive (`NCKH_PanDerm/runs/<RUN_NAME>`). Nếu session bị ngắt, chạy lại từ đầu notebook với cùng `RUN_NAME`, run sẽ tự resume.
- Checkpoint tốt nhất chọn theo **macro recall trên validation** (`--monitor recall`). Đánh giá test thật chỉ làm một lần sau khi train xong, bằng `scripts/evaluate_inference.py --finetuned`.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd

if not Path('/usr/bin/nvidia-smi').is_file():
    raise RuntimeError('Runtime chưa có GPU. Chọn GPU T4 rồi chạy lại.')
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=True).stdout)

from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/NCKH_PanDerm')
DATA = ROOT / 'nckh_data'
MANIFESTS = ROOT / 'data/manifests'
CHECKPOINT = ROOT / 'checkpoints/panderm_bb_data6_checkpoint-499.pth'
TRAIN_CSV = MANIFESTS / 'isic2017_cls_trainphase.csv'
for path in (CHECKPOINT, TRAIN_CSV):
    if not path.is_file():
        raise FileNotFoundError(f'Thiếu {path}. Mount Drive và kiểm tra P3 trước.')

In [ ]:
REPO = Path('/content/nckh')
PANDERM = Path('/content/PanDerm')
PANDERM_COMMIT = 'fd7a80748ba7fc3e203fed88f909f4689d0d6f24'  # commit đã kiểm tra ở P2
for repository, url in ((REPO, 'https://github.com/NeuroDev204/nckh.git'), (PANDERM, 'https://github.com/SiyuanYan1/PanDerm.git')):
    if not repository.exists():
        subprocess.run(['git', 'clone', url, str(repository)], check=True)
    elif not (repository / '.git').is_dir():
        raise RuntimeError(f'{repository} tồn tại nhưng không phải repo Git.')
subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(PANDERM), 'checkout', PANDERM_COMMIT], check=True)

def apply_patch(patch: Path) -> None:
    # Idempotent: bỏ qua nếu bản vá đã áp dụng (kiểm tra bằng --reverse --check).
    base = ['git', '-C', str(PANDERM), 'apply', '--unidiff-zero']
    if subprocess.run([*base, '--reverse', '--check', str(patch)], capture_output=True).returncode == 0:
        print('Đã áp dụng:', patch.name)
        return
    subprocess.run([*base, str(patch)], check=True)
    print('Áp dụng bản vá:', patch.name)

apply_patch(REPO / 'patches/panderm_cls_resume.patch')      # resume giữ head/fc_norm
apply_patch(REPO / 'patches/panderm_cls_epoch_ckpt.patch')  # lưu checkpoint theo epoch + khôi phục metric tốt nhất

In [ ]:
# Môi trường model riêng (Python 3.10, torch 2.4.1), không thay torch của kernel.
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
MODEL_PYTHON = Path('/content/venv_nckh_cls/bin/python')
if not MODEL_PYTHON.is_file():
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '/content/venv_nckh_cls', '--python', '3.10'], check=True)
uv_install = [sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(MODEL_PYTHON)]
subprocess.run([*uv_install, 'torch==2.4.1', 'torchvision==0.19.1', '--index-url', 'https://download.pytorch.org/whl/cu118'], check=True)
subprocess.run([*uv_install, '-r', str(PANDERM / 'classification/requirements.txt'), '-e', str(REPO)], check=True)

## Chuẩn bị dữ liệu

Kiểm tra manifest trước khi train rồi chép đúng các ảnh được liệt kê sang đĩa local của runtime (đọc từ Drive từng ảnh rất chậm).

In [ ]:
manifest = pd.read_csv(TRAIN_CSV)
assert {'image', 'label', 'split'} <= set(manifest.columns), manifest.columns.tolist()
images_by_split = {name: set(group['image']) for name, group in manifest.groupby('split')}
# Chống mở nhầm test thật: trong file trainphase, split test phải đúng là bản sao của val.
assert images_by_split['test'] == images_by_split['val'], 'split test không phải bản sao val; dừng để bảo vệ test thật.'
assert not images_by_split['train'] & images_by_split['val'], 'train và val trùng ảnh.'
print(manifest.groupby(['split', 'label']).size().unstack(fill_value=0))

LOCAL_DATA = Path('/content/isic2017')
missing = []
for index, relative in enumerate(manifest['image'].unique(), start=1):
    source, target = DATA / 'ISIC2017' / relative, LOCAL_DATA / relative
    if not source.is_file():
        missing.append(relative)
        continue
    if not target.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)
    if index % 500 == 0:
        print('Đã chép', index, 'ảnh')
if missing:
    raise FileNotFoundError(f'Thiếu {len(missing)} ảnh trên Drive, ví dụ: {missing[:5]}')
LOCAL_CHECKPOINT = Path('/content') / CHECKPOINT.name
if not LOCAL_CHECKPOINT.is_file():
    shutil.copy2(CHECKPOINT, LOCAL_CHECKPOINT)
# Dataset của PanDerm ghép root + tên file bằng nối chuỗi nên root phải kết thúc bằng '/'.
ROOT_PATH = f'{LOCAL_DATA}/'

## Cấu hình run

Siêu tham số lấy từ `script/finetune_train.sh` của PanDerm (lr 5e-4, layer decay 0.65, drop path 0.2, mixup 0.8, cutmix 1.0, weighted sampler). Batch 64 × `update_freq` 2 = 128 hiệu dụng để vừa 16 GB của T4; chỉnh nếu pilot báo OOM. Đổi `PILOT = False` để chạy đủ epoch.

In [ ]:
PILOT = True
SEED = 122
EPOCHS = 1 if PILOT else 50
WARMUP_EPOCHS = 1 if PILOT else 10  # scheduler yêu cầu warmup <= tổng epoch
BATCH_SIZE = 64
UPDATE_FREQ = 2
RUN_NAME = f"cls-ft-{'pilot' if PILOT else 'full'}-seed{SEED}"
OUTPUT_DIR = ROOT / 'runs' / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

command = [
    str(MODEL_PYTHON), '-u', 'run_class_finetuning.py',
    '--model', 'PanDerm_Base_FT', '--pretrained_checkpoint', str(LOCAL_CHECKPOINT),
    '--nb_classes', '3', '--batch_size', str(BATCH_SIZE), '--update_freq', str(UPDATE_FREQ),
    '--lr', '5e-4', '--warmup_epochs', str(WARMUP_EPOCHS), '--epochs', str(EPOCHS),
    '--layer_decay', '0.65', '--drop_path', '0.2', '--weight_decay', '0.05',
    '--mixup', '0.8', '--cutmix', '1.0', '--weights', '--monitor', 'recall',
    '--sin_pos_emb', '--imagenet_default_mean_and_std', '--auto_resume',
    '--exp_name', RUN_NAME, '--output_dir', str(OUTPUT_DIR),
    '--csv_path', str(TRAIN_CSV), '--root_path', ROOT_PATH,
    '--seed', str(SEED), '--num_workers', '2',
]
# wandb được import và khởi tạo vô điều kiện trong script PanDerm; chạy offline để không cần tài khoản.
environment = {**os.environ, 'WANDB_MODE': 'offline', 'WANDB_DIR': '/content/wandb', 'PYTHONUNBUFFERED': '1'}
print('Run:', RUN_NAME, '->', OUTPUT_DIR)

In [ ]:
log_path = OUTPUT_DIR / 'train.log'
with log_path.open('a', encoding='utf-8') as log:
    log.write(f'\n===== start {pd.Timestamp.utcnow().isoformat()} =====\n')
    with subprocess.Popen(command, cwd=PANDERM / 'classification', env=environment,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
        if process.stdout is None:
            raise RuntimeError('Không mở được output tiến trình train.')
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        returncode = process.wait()
if returncode:
    raise subprocess.CalledProcessError(returncode, command)
print('Xong. Log:', log_path)

## Sau khi train

Ghi run card (hash manifest và checkpoint) và liệt kê checkpoint. Với run đủ epoch, bước tiếp theo là đánh giá test thật đúng một lần bằng `scripts/evaluate_inference.py --task cls --split test --finetuned <checkpoint-best.pth>`.

In [ ]:
print(sorted(path.name for path in OUTPUT_DIR.iterdir()))
best_metric = OUTPUT_DIR / 'best_metric.json'
print('Macro recall val tốt nhất:', json.loads(best_metric.read_text()) if best_metric.is_file() else 'chưa có')
subprocess.run([str(MODEL_PYTHON), '-m', 'nckh.runcard', str(OUTPUT_DIR), '--seed', str(SEED),
                '--input', f'pretrained={CHECKPOINT}', '--input', f'manifest={TRAIN_CSV}',
                '--config', f'epochs={EPOCHS}', '--config', f'batch_size={BATCH_SIZE}',
                '--config', f'update_freq={UPDATE_FREQ}', '--config', 'monitor=recall',
                '--config', f'pilot={PILOT}'], cwd=REPO, check=True)